In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from fairlearn.metrics import MetricFrame
from sklearn.metrics import (
                            accuracy_score,
                            balanced_accuracy_score,
                            precision_score,
                            recall_score,
                            f1_score,
                            roc_auc_score,
                            average_precision_score,
                            log_loss,
                            )
from fairlearn.metrics import (
                            MetricFrame,
                            false_positive_rate,
                            false_negative_rate,
                            selection_rate,
                            true_positive_rate,
                            true_negative_rate,
                            demographic_parity_difference,
                            selection_rate,
                            count
                        )

def zero_div_precision_score(y_true, y_pred):
    '''
    This parameter tells precision_score to return 0 instead of raising an error 
    when there are no positive predictions.
    '''
    return precision_score(y_true, y_pred, zero_division=0)

def evaluate_fairness_metrics(
    X_ohe: pd.DataFrame,
    y: pd.Series,
    X_protected: pd.DataFrame,
    model: BaseEstimator = None,
    metrics: dict = None,
    threshold: float = 0.5,
    test_size: float = 0.3,
    random_state: int = 42,
    drop_protected: bool = True,
    return_frame: bool = True,
):
    """
    Evaluate multiple fairness metrics across protected attributes using Fairlearn.

    Parameters:
    - X_ohe (pd.DataFrame): Feature matrix (may include protected columns).
    - y (pd.Series): Binary target.
    - X_protected (pd.DataFrame): Protected attributes (multiple columns allowed).
    - model (BaseEstimator): Any sklearn classifier. If None, uses DecisionTreeClassifier.
    - metrics (dict): Dictionary of metric name: function pairs.
    - threshold (float): Threshold for binarizing predict_proba output.
    - test_size (float): Fraction of data used for test set.
    - random_state (int): Random seed.
    - drop_protected (bool): Whether to drop one-hot encoded protected attributes from X_ohe.
    - return_frame (bool): Whether to return MetricFrame object(s).

    Returns:
    - Dict of metric results per protected attribute. Each entry includes:
      - overall: dict of overall scores
      - by_group: DataFrame of scores per group
      - (optional) metric_frame: Fairlearn MetricFrame object
    """
    if model is None:
        model = DecisionTreeClassifier(min_samples_leaf=10, max_depth=4, random_state=random_state)

    if metrics is None:
        metrics = {
            "accuracy": accuracy_score,  # % of correct predictions in the group
            "precision": zero_div_precision_score,  # % of predicted positives that are actually positive
            "recall": recall_score,  # % of actual positives that are correctly predicted (true positive rate)
            "false positive rate": false_positive_rate,  # % of actual negatives misclassified as positive (Type I error rate)
            "false negative rate": false_negative_rate,  # % of actual positives misclassified as negative (Type II error rate)
            "true positive rate" : true_positive_rate,  # % of actual positives correctly classified as positive (sensitivity or recall)
            "true negative rate" : true_negative_rate, # % of actual negatives correctly classified as negative (specificity)
            "selection rate": selection_rate,  # % predicted as positive regardless of ground truth (positive outcome rate)
            "count": count,  # Number of individuals in the group
        }

    results = {}

    if drop_protected:
        protected_prefixes = [col + "_" for col in X_protected.columns]
        cols_to_drop = [col for col in X_ohe.columns if any(col.startswith(prefix) for prefix in protected_prefixes)]
        X_features = X_ohe.drop(columns=cols_to_drop, errors='ignore')
    else:
        X_features = X_ohe.copy()

    for protected_col in X_protected.columns:
        A = X_protected[protected_col]

        X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
            X_features, y, A, test_size=test_size, random_state=random_state
        )

        model.fit(X_train, y_train)

        if hasattr(model, "predict_proba"):
            y_pred = (model.predict_proba(X_test)[:, 1] >= threshold).astype(int)
        else:
            y_pred = model.predict(X_test)

        mf = MetricFrame(metrics=metrics, y_true=y_test, y_pred=y_pred, sensitive_features=A_test)

        results[protected_col] = {
            "overall": mf.overall.to_dict(),
            "by_group": mf.by_group
        }

        if return_frame:
            results[protected_col]["metric_frame"] = mf

    return results

In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from fairlearn.reductions import ErrorRate, EqualizedOdds, ExponentiatedGradient
from fairlearn.metrics import MetricFrame
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
)
from fairlearn.metrics import (
    MetricFrame,
    false_positive_rate,
    false_negative_rate,
    selection_rate,
    true_positive_rate,
    true_negative_rate,
    demographic_parity_difference,
    selection_rate,
    count
)
# constraint
from fairlearn.reductions import (
    DemographicParity,             # Demographic Parity constraint
    EqualizedOdds,                 # Equalized Odds constraint
    TruePositiveRateParity,        # True Positive Rate Parity constraint
    FalsePositiveRateParity,       # False Positive Rate Parity constraint
    ErrorRateParity                # Error Rate Parity constraint
)

def mitigate_fairness_disparity(
    X_ohe: pd.DataFrame,
    y: pd.Series,
    X_protected: pd.DataFrame,
    model: BaseEstimator = None,
    constraint=None,
    objective=None,
    metrics: dict = None,
    test_size: float = 0.3,
    random_state: int = 42,
    drop_protected: bool = True,
    return_frame: bool = True,
):
    """
    Mitigate disparity across protected attributes using Exponentiated Gradient method.

    Parameters:
    - X_ohe (pd.DataFrame): Feature matrix (may include protected columns).
    - y (pd.Series): Binary target.
    - X_protected (pd.DataFrame): Protected attributes (multiple columns allowed).
    - model (BaseEstimator): Base classifier. If None, uses DecisionTreeClassifier.
    - constraint: Fairness constraint object (default : 'EqualizedOdds', 
        suggest option 'DemographicParity',    
                        'TruePositiveRateParity',       
                        'FalsePositiveRateParity',       
                        'ErrorRateParity')                
    - objective: Objective function (default : 'ErrorRate' Misclassification Error Rate ). 
    - metrics (dict): Dictionary of metric name: function pairs (e.g., accuracy, precision).
    - test_size (float): Fraction of data used for test set.
    - random_state (int): Random seed.
    - drop_protected (bool): Whether to drop one-hot encoded protected attributes from X_ohe.
    - return_frame (bool): Whether to return MetricFrame object(s).

    Returns:
    - Dict of metric results per protected attribute. Each entry includes:
        - overall: dict of overall metric scores
        - by_group: DataFrame of group-wise metric scores
        - (optional) metric_frame: Fairlearn MetricFrame object

    Note on `ExponentiatedGradient`:
    - This method does **not support `predict_proba`** or manual thresholding.
    - To influence decision thresholds, define a custom **cost-sensitive objective**:
        e.g., `ErrorRate(costs={"fp": 0.1, "fn": 0.9})`
    - This behaves like setting a threshold (e.g., 10%) by making false negatives more costly.
    """
    if model is None:
        model = DecisionTreeClassifier(min_samples_leaf=10, max_depth=4, random_state=random_state)

    if constraint is None:
        constraint = EqualizedOdds(difference_bound=0.01)

    if objective is None:
        objective = ErrorRate(costs={"fp": 0.1, "fn": 0.9})

    if metrics is None:
        metrics = {
            "accuracy": accuracy_score,
            "precision": zero_div_precision_score,
            "recall": recall_score,
            "false positive rate": false_positive_rate,
            "false negative rate": false_negative_rate,
            "true positive rate": true_positive_rate,
            "true negative rate": true_negative_rate,
            "selection rate": selection_rate,
            "count": count,
        }

    results = {}

    # Remove protected columns from input if needed 
    if drop_protected:
        protected_prefixes = [col + "_" for col in X_protected.columns]
        cols_to_drop = [col for col in X_ohe.columns if any(col.startswith(prefix) for prefix in protected_prefixes)]
        X_features = X_ohe.drop(columns=cols_to_drop, errors="ignore")
    else:
        X_features = X_ohe.copy()

    # Run mitigation for each protected attribute separately
    for protected_col in X_protected.columns:
        A = X_protected[protected_col]

        # Split data
        X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
            X_features, y, A, test_size=test_size, random_state=random_state
        )

        # Apply fairness mitigation
        mitigator = ExponentiatedGradient(model, constraints=constraint, objective=objective)
        mitigator.fit(X_train, y_train, sensitive_features=A_train)
        y_pred = mitigator.predict(X_test)

        # Calculate fairness metrics
        mf = MetricFrame(metrics=metrics, y_true=y_test, y_pred=y_pred, sensitive_features=A_test)

        results[protected_col] = {
            "overall": mf.overall.to_dict(),
            "by_group": mf.by_group
        }

        if return_frame:
            results[protected_col]["metric_frame"] = mf

    return results